# 05 — SIMCA mixture application

Objectif : appliquer aux images *mixtures* les configurations SIMCA figées en 04C.

Ce notebook est volontairement compact :
- aucune sélection de modèle ni calibration de seuil n'est réalisée ici ;
- toutes les tables nécessaires à l'analyse sont sauvegardées ;
- seulement **quatre figures essentielles au maximum** sont affichées pour une configuration de référence et une image difficile ;
- la génération exhaustive des graphiques est déléguée au script de reporting afin de conserver un notebook léger.

Sorties principales :
- métriques objet et pixel sur les mixtures ;
- prédictions objet binaires et décisions objet 3-way ;
- métriques par image ;
- diagnostics bord/cœur et sensibilité de la vérité terrain ;
- tables de confusion ;
- un petit ensemble de figures de synthèse.


In [1]:
from __future__ import annotations

import sys
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 260)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
    parse_preprocessing_steps,
    merge_config_metadata,
)

from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)
from src.spectra.preprocessing_configs import normalize_preprocessing_configs

from src.decision.labels import (
    predicted_col,
    true_col,
    UNCERTAIN_LABEL,
)
from src.decision.metrics import (
    add_binary_confusion_case,
    summarize_object_errors_by_image,
    summarize_pixel_errors_by_image,
    binary_confusion_table,
)
from src.decision.aggregation import object_threshold_grid
from src.decision.truth import add_pixel_truth_labels
from src.decision.confidence import (
    add_binary_object_confidence,
    add_binary_pixel_confidence,
)
from src.decision.maps import assign_object_decisions_to_pixels
from src.decision.uncertainty import (
    evaluate_three_way_by_config,
    add_three_way_confidence,
    three_way_confusion_table,
)
from src.decision.border import summarize_border_diagnostics_by_config

from src.workflows.simca import (
    make_target_train_filters,
    refit_selected_simca_configs,
)
from src.workflows.simca_selection_utils import (
    ensure_candidate_columns,
    normalize_simca_rule_columns,
    fill_selected_config_defaults,
    summarize_parameter_tendencies,
)

from src.reporting.selection import (
    choose_images_for_config,
    choose_images_for_config_3way,
)

from src.visualization.plot_decision import (
    plot_three_way_confusion_heatmap,
    plot_binary_confusion_heatmap,
)
from src.visualization.plot_reporting import plot_mixture_diagnostic_panel
from src.visualization.plot_robustness import plot_truth_dilation_sensitivity
from src.visualization.common import save_figure_bundle, sanitize_filename

%load_ext autoreload
%autoreload 2


## 1. Configuration

In [3]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
USE_WAVELENGTH_WINDOW = False
WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Inputs from 04C
# ---------------------------------------------------------------------
RESULTS_04C_DIR = PROJECT_ROOT / "results" / f"04C_simca_pure_test_{RESULTS_TAG}"

FROZEN_REFERENCE_CONFIGS_PATH = (
    RESULTS_04C_DIR / "frozen_reference_configs.parquet"
)

PURE_TEST_METRICS_PATH = (
    RESULTS_04C_DIR / "pure_test_metrics.parquet"
)

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"05_simca_mixture_application_{RESULTS_TAG}"
DEBUG_DIR = RESULTS_DIR / "debug"
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MIXTURE_METRICS_PATH = RESULTS_DIR / "mixture_metrics.parquet"
MIXTURE_MODEL_SUMMARY_PATH = RESULTS_DIR / "mixture_model_summary.parquet"

MIXTURE_OBJECT_PREDICTIONS_PATH = RESULTS_DIR / "mixture_object_predictions.parquet"
MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "mixture_object_errors_by_image.parquet"
MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "mixture_pixel_errors_by_image.parquet"

MIXTURE_OBJECT_2WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_object_2way_confusion.parquet"
MIXTURE_PIXEL_2WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_pixel_2way_confusion.parquet"

MIXTURE_THREE_WAY_METRICS_PATH = RESULTS_DIR / "mixture_three_way_metrics.parquet"
MIXTURE_THREE_WAY_BY_IMAGE_PATH = RESULTS_DIR / "mixture_three_way_by_image.parquet"
MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH = RESULTS_DIR / "mixture_object_predictions_3way.parquet"
MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH = RESULTS_DIR / "mixture_pixel_predictions_3way_from_object_decision.parquet"

MIXTURE_OBJECT_3WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_object_3way_confusion.parquet"
MIXTURE_PIXEL_3WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_pixel_3way_confusion_from_object_decision.parquet"

MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH = RESULTS_DIR / "mixture_object_simca_q_t2_diagnostics.parquet"
MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH = RESULTS_DIR / "mixture_pixel_simca_q_t2_diagnostics.parquet"

MIXTURE_BORDER_DIAGNOSTIC_PATH = RESULTS_DIR / "mixture_border_diagnostic.parquet"
MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH = RESULTS_DIR / "mixture_truth_dilation_sensitivity.parquet"

MIXTURE_PARAMETER_TENDENCIES_PATH = RESULTS_DIR / "mixture_parameter_tendencies.parquet"
MIXTURE_APPLICATION_PROTOCOL_PATH = RESULTS_DIR / "mixture_application_protocol.parquet"
MIXTURE_REFIT_ERRORS_PATH = RESULTS_DIR / "mixture_refit_errors.parquet"

MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH = (
    DEBUG_DIR / "mixture_pixel_predictions_minimal.parquet"
)

# ---------------------------------------------------------------------
# Detection protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
NON_TARGET_LABEL = "almond"
UNCERTAIN_LABEL_USED = UNCERTAIN_LABEL

REFERENCE_CLASSES = ("almond", TARGET_CLASS)

# Final calibration for application:
# the model configurations were frozen in 04C, so all pure peanut batches
# can now be used to calibrate the final models before projecting mixtures.
MIXTURE_FINAL_TRAIN_BATCHES = [1, 2, 3, 4]

MIXTURE_FINAL_TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=MIXTURE_FINAL_TRAIN_BATCHES,
)

MIXTURE_FILTERS = {
    "sample_kind": ["mixture"],
}

# ---------------------------------------------------------------------
# Runtime flags
# ---------------------------------------------------------------------
RUN_MIXTURE_APPLICATION = True
APPLY_FIXED_THREE_WAY = True
RUN_BORDER_DIAGNOSTIC = True
RUN_TRUTH_DILATION_SENSITIVITY = True

# Full pixel tables are large. Keep False for the compact notebook.
# Set True only when an external workflow explicitly needs them.
SAVE_MIXTURE_PIXEL_TABLES = False

RANDOM_STATE = 42
REPLACE_BALANCED_PIXELS = False

CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

# Limit the number of frozen models for a quick debugging run.
# Keep None for the final application.
MAX_REFERENCE_CONFIGS = None

# Border/core diagnostic.
BORDER_DIAGNOSTIC_WIDTHS = [1, 2, 3]

# Prediction is fixed; only the approximate mixture truth map is recomputed.
TRUTH_DILATION_RADII = [0, 1, 2, 3, 4, 5]

# ---------------------------------------------------------------------
# Compact visualisation protocol
# ---------------------------------------------------------------------
RUN_ESSENTIAL_VISUALIZATION = True
SHOW_ESSENTIAL_FIGURES_INLINE = True
SAVE_ESSENTIAL_FIGURES = False
ESSENTIAL_FIGURE_FORMATS = ("html",)

# None = use the first frozen reference configuration passing the guardrail.
DIAGNOSTIC_CONFIG_ID = None
N_DIAGNOSTIC_IMAGES = 1

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_04C_DIR:", RESULTS_04C_DIR)
print("FROZEN_REFERENCE_CONFIGS_PATH:", FROZEN_REFERENCE_CONFIGS_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("MIXTURE_FINAL_TRAIN_FILTERS:", MIXTURE_FINAL_TRAIN_FILTERS)
print("MIXTURE_FILTERS:", MIXTURE_FILTERS)

DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_04C_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all
FROZEN_REFERENCE_CONFIGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\frozen_reference_configs.parquet
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all
MIXTURE_FINAL_TRAIN_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['peanut'], 'batch': [1, 2, 3, 4]}
MIXTURE_FILTERS: {'sample_kind': ['mixture']}


## 2. Load database and frozen configurations

In [4]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

if not FROZEN_REFERENCE_CONFIGS_PATH.exists():
    raise FileNotFoundError(
        f"Frozen reference configs not found: {FROZEN_REFERENCE_CONFIGS_PATH}. "
        "Run notebook 04C first."
    )

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

# ---------------------------------------------------------------------
# Wavelength handling
# ---------------------------------------------------------------------
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)

else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

reference_configs_df = load_parquet(FROZEN_REFERENCE_CONFIGS_PATH)

if PURE_TEST_METRICS_PATH.exists():
    pure_test_metrics_df = load_parquet(PURE_TEST_METRICS_PATH)
else:
    pure_test_metrics_df = pd.DataFrame()

reference_configs_df = ensure_candidate_columns(reference_configs_df)
reference_configs_df = normalize_simca_rule_columns(reference_configs_df)
reference_configs_df = fill_selected_config_defaults(
    reference_configs_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": 40,
        "alpha": 0.05,
        "object_threshold": 0.75,
    },
)
reference_configs_df = reference_configs_df.copy()
reference_configs_df["selected_config_id"] = reference_configs_df["selected_config_id"].astype(str)

# Les modèles sont déjà figés en 04C.
# On préserve donc leur ordre et leur rang de sélection.
if "frozen_reference_rank" in reference_configs_df.columns:
    reference_configs_df = (
        reference_configs_df
        .sort_values("frozen_reference_rank")
        .reset_index(drop=True)
    )
else:
    reference_configs_df["frozen_reference_rank"] = np.arange(
        1,
        len(reference_configs_df) + 1,
    )

# Keep only models explicitly declared ready for mixture application.
# Do not use astype(bool): the string "False" would otherwise evaluate to True.
if "ready_for_mixture_application" in reference_configs_df.columns:
    ready_raw = reference_configs_df["ready_for_mixture_application"]
    ready_mask = (
        ready_raw.eq(True)
        | ready_raw.astype(str).str.strip().str.lower().isin(
            {"true", "1", "yes", "y"}
        )
    )
    reference_configs_df = reference_configs_df[ready_mask].copy()

required_threshold_cols = [
    "selected_config_id",
    "object_threshold",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
]

missing_threshold_cols = [
    col for col in required_threshold_cols
    if col not in reference_configs_df.columns
]

if missing_threshold_cols:
    raise KeyError(
        "Missing fixed decision thresholds in frozen_reference_configs_df: "
        f"{missing_threshold_cols}. Re-run 04C with fixed 3-way threshold export."
    )

if reference_configs_df.empty:
    raise RuntimeError(
        "No frozen reference configuration is ready for mixture application."
    )

missing_threshold_values = reference_configs_df[required_threshold_cols].isna().any(axis=1)
if missing_threshold_values.any():
    bad_ids = reference_configs_df.loc[
        missing_threshold_values,
        "selected_config_id",
    ].astype(str).tolist()
    raise ValueError(
        "Some frozen configurations contain missing decision thresholds: "
        f"{bad_ids}"
    )

if MAX_REFERENCE_CONFIGS is not None:
    reference_configs_df = (
        reference_configs_df
        .head(int(MAX_REFERENCE_CONFIGS))
        .copy()
        .reset_index(drop=True)
    )

print("Database loaded")
print("n objects:", len(object_db))
print("n images:", len(image_db))
print("n active bands:", len(wavelengths))
print("Frozen reference configs:", reference_configs_df.shape)
print("Pure test metrics:", pure_test_metrics_df.shape)

display(wavelength_config_df)
display(reference_configs_df.head(30))

Database loaded
n objects: 1262
n images: 48
n active bands: 63
Frozen reference configs: (16, 100)
Pure test metrics: (31, 114)


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,selected_config_id,frozen_reference_rank,ready_for_mixture_application,candidate_source,candidate_source_file,candidate_file_source_kind,candidate_input_rank,selection_split,selection_strategy,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,sg_window_length,sg_polyorder,position_dilation_radius,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,mean_fn_rate,std_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,max_fp_rate,mean_balanced_accuracy,is_robust_2way_pareto,is_robust_3way_pareto,robust_pareto_axis,optuna_trial_number,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_mean,fp_rate_max,fp_rate_std,balanced_accuracy_mean,object_threshold_median,pure_test_n,pure_test_tp,pure_test_fn,pure_test_fp,pure_test_tn,pure_test_balanced_accuracy,pure_test_target_sensitivity,pure_test_non_target_specificity,pure_test_fn_rate,pure_test_fp_rate,pure_test_f1_score,pure_test_accuracy,pure_test_precision,pure_test_selection_score,pure_test_3way_n,pure_test_3way_n_target,pure_test_3way_n_non_target,pure_test_3way_target_miss_rate,pure_test_3way_screening_sensitivity,pure_test_3way_non_target_false_accept_rate,pure_test_3way_uncertain_rate,pure_test_3way_coverage_rate,pure_test_3way_non_target_auto_reject_rate,pure_test_3way_decided_balanced_accuracy,passes_pure_test_binary_guardrail,passes_pure_test_3way_guardrail,passes_pure_test_guardrail,pure_test_guardrail_reason,rule_original,rule_variant_original,rule_token
0,optuna_object_matrix_0149,1,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,1,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,peanut,almond,5,0.01,0.50,0.50,0.95,15,2,4,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.345455,0.509259,0.490741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,149.0,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.536364,0.50,77,29,0,21,27,0.781250,1.000000,0.562500,0.000000,0.437500,0.734177,0.727273,0.580000,-0.386246,77,29,48,0.000000,1.000000,0.000000,0.597403,0.402597,0.562500,1.000000,True,True,True,pass,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv
1,optuna_object_matrix_0081,2,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,2,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,5,0.01,0.50,0.50,0.95,15,2,5,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.290909,0.546296,0.453704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,81.0,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.536364,0.50,77,29,0,21,27,0.781250,1.000000,0.562500,0.000000,0.437500,0.734177,0.727273,0.580000,-0.386246,77,29,48,0.000000,1.000000,0.000000,0.610390,0.389610,0.562500,1.000000,True,True,True,pass,data_driven,data_driven_emp_cv,data_driven_emp_cv
2,optuna_object_matrix_0148,3,True,04B2_optuna_challenge,C:\Use

In [5]:
expected_parser_output = ["absorbance", "snv", "sg_d1"]
actual_parser_output = parse_preprocessing_steps("absorbance_snv_sg_d1")
if actual_parser_output != expected_parser_output:
    raise RuntimeError(
        "src.utils.parse_preprocessing_steps is not the corrected version. "
        f"Expected {expected_parser_output}, got {actual_parser_output}."
    )

PREPROCESSING_CONFIGS = {
    str(row["preprocessing"]): tuple(
        parse_preprocessing_steps(row["preprocessing_steps"])
    )
    for _, row in (
        reference_configs_df
        .drop_duplicates("preprocessing")
        .iterrows()
    )
}

PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

print("Preprocessing configs used for final mixture refit:")
display(
    pd.DataFrame(
        [
            {
                "preprocessing": name,
                "preprocessing_steps": "+".join(steps),
            }
            for name, steps in PREPROCESSING_CONFIGS.items()
        ]
    )
)


Preprocessing configs used for final mixture refit:


,preprocessing,preprocessing_steps
0,sg_smooth,sg_smooth
1,absorbance_sg_smooth,absorbance+sg_smooth
2,absorbance_sg_d2,absorbance+sg_d2
3,raw,raw
4,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth


## 3. Apply frozen SIMCA reference models to mixture images

In [6]:
REFERENCE_METADATA_COLS = [
    "candidate_source",
    "candidate_input_rank",
    "frozen_reference_rank",
    "ready_for_mixture_application",
    "selection_split",
    "selection_strategy",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
    "validation_fn_rate",
    "validation_fp_rate",
    "validation_3way_target_miss_rate",
    "validation_3way_non_target_false_accept_rate",
    "validation_3way_uncertain_rate",
    "validation_3way_coverage_rate",
    "pure_test_fn_rate",
    "pure_test_fp_rate",
    "pure_test_3way_target_miss_rate",
    "pure_test_3way_non_target_false_accept_rate",
    "pure_test_3way_uncertain_rate",
    "pure_test_3way_coverage_rate",
    "passes_pure_test_guardrail",
    "pure_test_guardrail_reason",
]

In [7]:
if not RUN_MIXTURE_APPLICATION:
    raise RuntimeError(
        "RUN_MIXTURE_APPLICATION=False is not recommended because intermediate "
        "pixel/object tables are not saved by default."
    )

(
    mixture_metrics_df,
    mixture_objects_df,
    mixture_pixels_df,
    mixture_pixel_errors_by_image_df,
    mixture_refit_errors_df,
) = refit_selected_simca_configs(
    selected_configs_df=reference_configs_df,
    object_db=object_db,
    image_db=image_db,
    train_filters=MIXTURE_FINAL_TRAIN_FILTERS,
    projection_filters=MIXTURE_FILTERS,
    preprocessing_configs=PREPROCESSING_CONFIGS,
    evaluation_split="mixture_application",
    wavelengths=wavelengths,
    random_state=RANDOM_STATE,
    replace=REPLACE_BALANCED_PIXELS,
    cv_n_splits=CV_N_SPLITS,
    cv_group_col=CV_GROUP_COL,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

mixture_objects_df = add_binary_confusion_case(
    mixture_objects_df,
    target_class=TARGET_CLASS,
    level="object",
)

mixture_pixels_df = add_binary_confusion_case(
    mixture_pixels_df,
    target_class=TARGET_CLASS,
    level="pixel",
)

mixture_objects_df = merge_config_metadata(
    mixture_objects_df,
    reference_configs_df,
    id_col="selected_config_id",
    columns=REFERENCE_METADATA_COLS,
)

mixture_pixels_df = merge_config_metadata(
    mixture_pixels_df,
    reference_configs_df,
    id_col="selected_config_id",
    columns=REFERENCE_METADATA_COLS,
)

save_parquet(mixture_metrics_df, MIXTURE_METRICS_PATH)
save_parquet_if_nonempty(mixture_refit_errors_df, MIXTURE_REFIT_ERRORS_PATH)

print("Mixture metrics:", mixture_metrics_df.shape)
print("Mixture objects:", mixture_objects_df.shape)
print("Mixture pixels:", mixture_pixels_df.shape)
print("Refit errors:", mixture_refit_errors_df.shape)
print("Saved:", MIXTURE_METRICS_PATH)

display(mixture_metrics_df.head(30))
display(mixture_refit_errors_df)

[mixture_application] optuna_object_matrix_0149


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


[mixture_application] optuna_object_matrix_0081


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


[mixture_application] optuna_object_matrix_0148


...

[mixture_application] 04A_pixel_matrix_0005


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Mixture metrics: (16, 117)
Mixture objects: (11552, 73)
Mixture pixels: (1020352, 96)
Refit errors: (0, 0)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_metrics.parquet


,selected_config_id,frozen_reference_rank,ready_for_mixture_application,candidate_source,candidate_source_file,candidate_file_source_kind,candidate_input_rank,selection_split,selection_strategy,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,sg_window_length,sg_polyorder,position_dilation_radius,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,mean_fn_rate,std_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,max_fp_rate,mean_balanced_accuracy,is_robust_2way_pareto,is_robust_3way_pareto,robust_pareto_axis,optuna_trial_number,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_mean,fp_rate_max,fp_rate_std,balanced_accuracy_mean,object_threshold_median,pure_test_n,pure_test_tp,pure_test_fn,pure_test_fp,pure_test_tn,pure_test_balanced_accuracy,pure_test_target_sensitivity,pure_test_non_target_specificity,pure_test_fn_rate,pure_test_fp_rate,pure_test_f1_score,pure_test_accuracy,pure_test_precision,pure_test_selection_score,pure_test_3way_n,pure_test_3way_n_target,pure_test_3way_n_non_target,pure_test_3way_target_miss_rate,pure_test_3way_screening_sensitivity,pure_test_3way_non_target_false_accept_rate,pure_test_3way_uncertain_rate,pure_test_3way_coverage_rate,pure_test_3way_non_target_auto_reject_rate,pure_test_3way_decided_balanced_accuracy,passes_pure_test_binary_guardrail,passes_pure_test_3way_guardrail,passes_pure_test_guardrail,pure_test_guardrail_reason,rule_original,rule_variant_original,rule_token,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,evaluation_split,n_projected_objects,n_projected_pixels
0,optuna_object_matrix_0149,1,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,1,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,peanut,almond,5.0,0.01,0.50,0.50,0.95,15,2,4,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.345455,0.509259,0.490741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,149.0,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.536364,0.50,77,29,0,21,27,0.781250,1.000000,0.562500,0.000000,0.437500,0.734177,0.727273,0.580000,-0.386246,77,29,48,0.000000,1.000000,0.000000,0.597403,0.402597,0.562500,1.000000,True,True,True,pass,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,almond,722,139,7,283,293,0.952055,0.508681,0.730368,0.598338,0.329384,0.489437,0.047945,0.491319,mixture_application,722,63772
1,optuna_object_matrix_0081,2,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,2,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,5.0,0.01,0.50,0.50,0.95,15,2,5,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.290909,0.546296,0.453704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,81.0,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.0000

""


## 4. Two-way performance summaries on mixture images

In [8]:
MODEL_GROUP_COLS = [
    "selected_config_id",
    "selection_strategy",
    "matrix_family",
    "training_matrix_id",
    "model_family",
    "matrix_method",
    "preprocessing",
    "selected_rule_name",
    "rule",
    "rule_variant",
    "rule_for_refit",
    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "m",
    "m_effective",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
]

MODEL_GROUP_COLS = [
    col for col in MODEL_GROUP_COLS
    if col in mixture_objects_df.columns
    and col in mixture_pixels_df.columns
]

mixture_object_by_model_df = summarize_object_errors_by_image(
    mixture_objects_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=MODEL_GROUP_COLS,
)

mixture_pixel_by_model_df = summarize_pixel_errors_by_image(
    mixture_pixels_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=MODEL_GROUP_COLS,
)

print("Object metrics by model:", mixture_object_by_model_df.shape)
print("Pixel metrics by model:", mixture_pixel_by_model_df.shape)

display(
    mixture_object_by_model_df
    .sort_values(["fn_rate", "fp_rate", "balanced_accuracy"], ascending=[True, True, False])
    .head(30)
)

display(
    mixture_pixel_by_model_df
    .sort_values(["fn_rate", "fp_rate", "balanced_accuracy"], ascending=[True, True, False])
    .head(30)
)

Object metrics by model: (16, 41)
Pixel metrics by model: (16, 41)


,selected_config_id,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_objects,object_accuracy,object_balanced_accuracy,object_fn_rate,object_fp_rate
15,optuna_pixel_matrix_0285,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_smooth,data_driven_chi2,data_driven,data_driven_chi2,data_driven_chi2,7.0,0.01,0.60,15,2,2,40,NaN,center,NaN,peanut,almond,722,146,0,142,434,1.000000,0.753472,0.876736,0.803324,0.506944,0.672811,0.000000,0.246528,722,0.803324,0.876736,0.000000,0.246528
14,optuna_pixel_matrix_0257,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_snv_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,6.0,0.05,0.55,15,2,3,80,NaN,center,NaN,peanut,almond,722,145,1,137,439,0.993151,0.762153,0.877652,0.808864,0.514184,0.677570,0.006849,0.237847,722,0.808864,0.877652,0.006849,0.237847
13,04A_pixel_matrix_0001,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,NaN,0.01,0.75,11,2,3,40,40.0,random,random,peanut,almond,722,145,1,318,258,0.993151,0.447917,0.720534,0.558172,0.313175,0.476190,0.006849,0.552083,722,0.558172,0.720534,0.006849,0.552083
12,04A_pixel_matrix_0002,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_center_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_emp_cv,simple,simple_emp_cv,simple_emp_cv,NaN,0.05,0.70,11,2,3,40,40.0,center,center,peanut,almond,722,144,2,192,384,0.986301,0.666667,0.826484,0.731302,0.428571,0.597510,0.013699,0.333333,722,0.731302,0.826484,0.013699,0.333333
11,optuna_pixel_matrix_0083,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,6.0,0.05,0.55,15,2,3,20,NaN,random,NaN,peanut,almond,722,143,3,75,501,0.979452,0.869792,0.924622,0.891967,0.655963,0.785714,0.020548,0.130208,722,0.891967,0.924622,0.020548,0.130208
10,optuna_pixel_matrix_0111,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_smooth,data_driven_chi2,data_driven,data_driven_chi2,data_driven_chi2,7.0,0.01,0.75,11,2,3,80,NaN,center,NaN,peanut,almond,722,143,3,89,487,0.979452,0.845486,0.912469,0.872576,0.616379,0.756614,0.020548,0.154514,722,0.872576,0.912469,0.020548,0.154514
9,04A_pixel_matrix_0004,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,NaN,0.01,0.80,11,2,3,40,40.0,random,random,peanut,almond,722,142,4,262,314,0.972603,0.545139,0.758871,0.631579,0.351485,0.516364,0.027397,0.454861,722,0.631579,0.758871,0.027397,0.454861
8,optuna_object_matrix_0081,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,5.0,0.01,0.50,15,2,5,40,NaN,random,NaN,peanut,almond,722,139,7,279,297,0.952055,0.515625,0.733840,0.603878,0.332536,0.492908,0.047945,0.484375,722,0.603878,0.733840,0.047945,0.484375
7,optuna_object_matrix_0149,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_chi2_emp_cv,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,5.0,0.01,0.50,15,2,4

,selected_config_id,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_pixels,pixel_accuracy,pixel_balanced_accuracy,pixel_fn_rate,pixel_fp_rate
14,04A_pixel_matrix_0001,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,NaN,0.01,0.75,11,2,3,40,40.0,random,random,peanut,almond,63772,9880,434,37127,16331,0.957921,0.305492,0.631707,0.411011,0.210181,0.344725,0.042079,0.694508,63772,0.411011,0.631707,0.042079,0.694508
15,04A_pixel_matrix_0004,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,NaN,0.01,0.80,11,2,3,40,40.0,random,random,peanut,almond,63772,9880,434,37127,16331,0.957921,0.305492,0.631707,0.411011,0.210181,0.344725,0.042079,0.694508,63772,0.411011,0.631707,0.042079,0.694508
13,optuna_pixel_matrix_0111,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_smooth,data_driven_chi2,data_driven,data_driven_chi2,data_driven_chi2,7.0,0.01,0.75,11,2,3,80,NaN,center,NaN,peanut,almond,63772,9831,483,19332,34126,0.953170,0.638370,0.795770,0.689284,0.337105,0.498062,0.046830,0.361630,63772,0.689284,0.795770,0.046830,0.361630
12,optuna_pixel_matrix_0285,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_smooth,data_driven_chi2,data_driven,data_driven_chi2,data_driven_chi2,7.0,0.01,0.60,15,2,2,40,NaN,center,NaN,peanut,almond,63772,9616,698,18033,35425,0.932325,0.662670,0.797497,0.706282,0.347788,0.506599,0.067675,0.337330,63772,0.706282,0.797497,0.067675,0.337330
11,04A_pixel_matrix_0002,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_center_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_emp_cv,simple,simple_emp_cv,simple_emp_cv,NaN,0.05,0.70,11,2,3,40,40.0,center,center,peanut,almond,63772,9477,837,27208,26250,0.918848,0.491040,0.704944,0.560230,0.258334,0.403285,0.081152,0.508960,63772,0.560230,0.704944,0.081152,0.508960
10,04A_pixel_matrix_0005,04A_grid_rule_variant_universe__04B_robust_par...,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_snv_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,NaN,0.05,0.75,11,2,3,40,40.0,random,random,peanut,almond,63772,9249,1065,19018,34440,0.896742,0.644244,0.770493,0.685081,0.327201,0.479459,0.103258,0.355756,63772,0.685081,0.770493,0.103258,0.355756
9,optuna_pixel_matrix_0257,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_snv_sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,6.0,0.05,0.55,15,2,3,80,NaN,center,NaN,peanut,almond,63772,9146,1168,15974,37484,0.886756,0.701186,0.793971,0.731199,0.364092,0.516227,0.113244,0.298814,63772,0.731199,0.793971,0.113244,0.298814
8,optuna_pixel_matrix_0083,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,6.0,0.05,0.55,15,2,3,20,NaN,random,NaN,peanut,almond,63772,9105,1209,13194,40264,0.882781,0.753189,0.817985,0.774149,0.408314,0.558366,0.117219,0.246811,63772,0.774149,0.817985,0.117219,0.246811
7,optuna_object_matrix_0149,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_ch

In [9]:
KEY_COLS = [
    col for col in MODEL_GROUP_COLS
    if col in mixture_object_by_model_df.columns
    and col in mixture_pixel_by_model_df.columns
]

GENERIC_METRIC_COLS = [
    "n",
    "tp",
    "fn",
    "fp",
    "tn",
    "target_sensitivity",
    "non_target_specificity",
    "balanced_accuracy",
    "accuracy",
    "precision",
    "f1_score",
    "fn_rate",
    "fp_rate",
]

object_metric_cols = [
    col for col in GENERIC_METRIC_COLS
    if col in mixture_object_by_model_df.columns
]

pixel_metric_cols = [
    col for col in GENERIC_METRIC_COLS
    if col in mixture_pixel_by_model_df.columns
]

object_prefixed = (
    mixture_object_by_model_df[KEY_COLS + object_metric_cols]
    .rename(columns={col: f"object_{col}" for col in object_metric_cols})
)

pixel_prefixed = (
    mixture_pixel_by_model_df[KEY_COLS + pixel_metric_cols]
    .rename(columns={col: f"pixel_{col}" for col in pixel_metric_cols})
)

mixture_model_summary_df = object_prefixed.merge(
    pixel_prefixed,
    on=KEY_COLS,
    how="outer",
)

# Diagnostic only: this score helps inspect mixture behaviour, but it is not used
# to select or re-rank the final reference models.
mixture_model_summary_df["mixture_reference_score"] = (
    -20.0 * mixture_model_summary_df["object_fn_rate"].fillna(1.0)
    -3.0 * mixture_model_summary_df["object_fp_rate"].fillna(1.0)
    -2.0 * mixture_model_summary_df["pixel_fn_rate"].fillna(1.0)
    -1.0 * mixture_model_summary_df["pixel_fp_rate"].fillna(1.0)
    +2.0 * mixture_model_summary_df["object_balanced_accuracy"].fillna(0.0)
    +0.5 * mixture_model_summary_df["pixel_balanced_accuracy"].fillna(0.0)
)

sort_cols = [
    "object_fn_rate",
    "pixel_fn_rate",
    "object_fp_rate",
    "pixel_fp_rate",
    "mixture_reference_score",
]

sort_cols = [col for col in sort_cols if col in mixture_model_summary_df.columns]

ascending = [
    True if col != "mixture_reference_score" else False
    for col in sort_cols
]

mixture_model_summary_df = (
    mixture_model_summary_df
    .sort_values(sort_cols, ascending=ascending)
    .reset_index(drop=True)
)

save_parquet(mixture_model_summary_df, MIXTURE_MODEL_SUMMARY_PATH)

print("Mixture model summary:", mixture_model_summary_df.shape)
print("Saved:", MIXTURE_MODEL_SUMMARY_PATH)

display(
    mixture_model_summary_df[
        [
            col for col in [
                "selected_config_id",
                "matrix_family",
                "training_matrix_id",
                "model_family",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
                "object_fn",
                "object_fp",
                "object_balanced_accuracy",
                "object_fn_rate",
                "object_fp_rate",
                "pixel_fn",
                "pixel_fp",
                "pixel_balanced_accuracy",
                "pixel_fn_rate",
                "pixel_fp_rate",
                "mixture_reference_score",
            ]
            if col in mixture_model_summary_df.columns
        ]
    ].head(40)
)

Mixture model summary: (16, 48)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_model_summary.parquet


,selected_config_id,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,object_fn,object_fp,object_balanced_accuracy,object_fn_rate,object_fp_rate,pixel_fn,pixel_fp,pixel_balanced_accuracy,pixel_fn_rate,pixel_fp_rate,mixture_reference_score
0,optuna_pixel_matrix_0285,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_smooth,data_driven_chi2,7.0,0.01,0.60,0,142,0.876736,0.000000,0.246528,698,18033,0.797497,0.067675,0.337330,0.939957
1,04A_pixel_matrix_0001,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_chi2,NaN,0.01,0.75,1,318,0.720534,0.006849,0.552083,434,37127,0.631707,0.042079,0.694508,-0.814981
2,optuna_pixel_matrix_0257,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_snv_sg_smooth,simple_chi2,6.0,0.05,0.55,1,137,0.877652,0.006849,0.237847,1168,15974,0.793971,0.113244,0.298814,0.776459
3,04A_pixel_matrix_0002,pixel_matrix,balanced_pixel_center_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_emp_cv,NaN,0.05,0.70,2,192,0.826484,0.013699,0.333333,837,27208,0.704944,0.081152,0.508960,0.060203
4,optuna_pixel_matrix_0111,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_smooth,data_driven_chi2,7.0,0.01,0.75,3,89,0.912469,0.020548,0.154514,483,19332,0.795770,0.046830,0.361630,0.893034
5,optuna_pixel_matrix_0083,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,sg_smooth,simple_chi2,6.0,0.05,0.55,3,75,0.924622,0.020548,0.130208,1209,13194,0.817985,0.117219,0.246811,0.975403
6,04A_pixel_matrix_0004,pixel_matrix,balanced_pixel_random_m40,empirical_cv_rule,balanced_pixels,absorbance_sg_smooth,simple_chi2,NaN,0.01,0.80,4,262,0.758871,0.027397,0.454861,434,37127,0.631707,0.042079,0.694508,-0.857599
7,optuna_object_matrix_0149,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_chi2_emp_cv,5.0,0.01,0.50,7,283,0.730368,0.047945,0.491319,1721,25151,0.681329,0.166861,0.470481,-1.435665
8,optuna_object_matrix_0081,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,data_driven_emp_cv,5.0,0.01,0.50,7,279,0.733840,0.047945,0.484375,1745,24964,0.681915,0.169188,0.466983,-1.408751
9,optuna_object_matrix_0148,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,5.0,0.01,0.50,7,300,0.715611,0.047945,0.520833,1959,25606,0.665536,0.189936,0.478993,-1.616280


In [10]:
IMAGE_GROUP_COLS_OBJECT = MODEL_GROUP_COLS + ["source_image"]
IMAGE_GROUP_COLS_OBJECT = [
    col for col in IMAGE_GROUP_COLS_OBJECT
    if col in mixture_objects_df.columns
]

IMAGE_GROUP_COLS_PIXEL = MODEL_GROUP_COLS + ["source_image"]
IMAGE_GROUP_COLS_PIXEL = [
    col for col in IMAGE_GROUP_COLS_PIXEL
    if col in mixture_pixels_df.columns
]

mixture_object_errors_by_image_df = summarize_object_errors_by_image(
    mixture_objects_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=IMAGE_GROUP_COLS_OBJECT,
)
mixture_object_errors_by_image_df[
    "n_true_target_objects"
] = (
    mixture_object_errors_by_image_df["tp"].fillna(0)
    + mixture_object_errors_by_image_df["fn"].fillna(0)
).astype(int)

mixture_pixel_errors_by_image_df = summarize_pixel_errors_by_image(
    mixture_pixels_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=IMAGE_GROUP_COLS_PIXEL,
)

save_parquet(mixture_object_errors_by_image_df, MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH)
save_parquet(mixture_pixel_errors_by_image_df, MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH)

print("Object errors by image:", mixture_object_errors_by_image_df.shape)
print("Pixel errors by image:", mixture_pixel_errors_by_image_df.shape)
print("Saved:")
print(" -", MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH)
print(" -", MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH)

display(mixture_object_errors_by_image_df.head(30))
display(mixture_pixel_errors_by_image_df.head(30))

Object errors by image: (320, 43)
Pixel errors by image: (320, 42)
Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_errors_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_pixel_errors_by_image.parquet


,selected_config_id,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,source_image,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_objects,object_accuracy,object_balanced_accuracy,object_fn_rate,object_fp_rate,n_true_target_objects
0,04A_object_matrix_0007,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,raw,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,alm4pea2,peanut,almond,31,0,1,10,20,0.000000,0.666667,0.333333,0.645161,0.000000,NaN,1.000000,0.333333,31,0.645161,0.333333,1.000000,0.333333,1
1,optuna_object_matrix_0081,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,5.0,0.01,0.50,15,2,5,40,NaN,random,NaN,alm5pea3,peanut,almond,38,0,1,9,28,0.000000,0.756757,0.378378,0.736842,0.000000,NaN,1.000000,0.243243,38,0.736842,0.378378,1.000000,0.243243,1
2,optuna_object_matrix_0148,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,alternative,alternative_empHQ_fixed2,alternative_empHQ_fixed2,5.0,0.01,0.50,15,2,3,40,NaN,random,NaN,alm5pea3,peanut,almond,38,0,1,9,28,0.000000,0.756757,0.378378,0.736842,0.000000,NaN,1.000000,0.243243,38,0.736842,0.378378,1.000000,0.243243,1
3,optuna_object_matrix_0149,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_chi2_emp_cv,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,5.0,0.01,0.50,15,2,4,40,NaN,random,NaN,alm5pea3,peanut,almond,38,0,1,9,28,0.000000,0.756757,0.378378,0.736842,0.000000,NaN,1.000000,0.243243,38,0.736842,0.378378,1.000000,0.243243,1
4,04A_object_matrix_0007,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,raw,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,alm5pea2,peanut,almond,33,0,1,3,29,0.000000,0.906250,0.453125,0.878788,0.000000,NaN,1.000000,0.093750,33,0.878788,0.453125,1.000000,0.093750,1
5,04A_object_matrix_0007,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,raw,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,alm5pea1,peanut,almond,32,0,1,2,29,0.000000,0.935484,0.467742,0.906250,0.000000,NaN,1.000000,0.064516,32,0.906250,0.467742,1.000000,0.064516,1
6,04A_object_matrix_0003,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,alm5pea1,peanut,almond,32,0,1,1,30,0.000000,0.967742,0.483871,0.937500,0.000000,NaN,1.000000,0.032258,32,0.937500,0.483871,1.000000,0.032258,1
7,04A_object_matrix_0003,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,alm5pea2,peanut,almond,33,0,1,1,31,0.000000,0.968750,0.484375,0.939394,0.000000,NaN,1.000000,0.031250,33,0.939394,0.484375,1.000000,0.031250,1
8,04A_object_matrix_0004,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,

,selected_config_id,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,source_image,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_pixels,pixel_accuracy,pixel_balanced_accuracy,pixel_fn_rate,pixel_fp_rate
0,optuna_object_matrix_0148,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,alternative,alternative_empHQ_fixed2,alternative_empHQ_fixed2,5.0,0.01,0.50,15,2,3,40,NaN,random,NaN,alm5pea3,peanut,almond,4519,80,144,1543,2752,0.357143,0.640745,0.498944,0.626687,0.049291,0.086627,0.642857,0.359255,4519,0.626687,0.498944,0.642857,0.359255
1,04A_object_matrix_0007,04A_grid_rule_variant_universe__04B_robust_par...,object_matrix,object_median,empirical_cv_rule,object_median,raw,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,alm5pea3,peanut,almond,4519,87,137,1512,2783,0.388393,0.647963,0.518178,0.635096,0.054409,0.095447,0.611607,0.352037,4519,0.635096,0.518178,0.611607,0.352037
2,optuna_object_matrix_0149,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_chi2_emp_cv,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,5.0,0.01,0.50,15,2,4,40,NaN,random,NaN,alm5pea3,peanut,almond,4519,89,135,1456,2839,0.397321,0.661001,0.529161,0.647931,0.057605,0.100622,0.602679,0.338999,4519,0.647931,0.529161,0.602679,0.338999
3,optuna_object_matrix_0081,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,5.0,0.01,0.50,15,2,5,40,NaN,random,NaN,alm5pea3,peanut,almond,4519,89,135,1440,2855,0.397321,0.664726,0.531024,0.651472,0.058208,0.101540,0.602679,0.335274,4519,0.651472,0.531024,0.602679,0.335274
4,optuna_pixel_matrix_0083,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,6.0,0.05,0.55,15,2,3,20,NaN,random,NaN,alm5pea3,peanut,almond,4519,100,124,533,3762,0.446429,0.875902,0.661165,0.854614,0.157978,0.233372,0.553571,0.124098,4519,0.854614,0.661165,0.553571,0.124098
5,optuna_pixel_matrix_0083,04B2_optuna_binary_pareto__3way_calibrated__04...,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,sg_smooth,simple_chi2,simple,simple_chi2,simple_chi2,6.0,0.05,0.55,15,2,3,20,NaN,random,NaN,alm5pea2,peanut,almond,4014,94,74,466,3380,0.559524,0.878835,0.719179,0.865471,0.167857,0.258242,0.440476,0.121165,4014,0.865471,0.719179,0.440476,0.121165
6,optuna_object_matrix_0148,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,alternative,alternative_empHQ_fixed2,alternative_empHQ_fixed2,5.0,0.01,0.50,15,2,3,40,NaN,random,NaN,alm5pea1,peanut,almond,3538,104,75,1259,2100,0.581006,0.625186,0.603096,0.622951,0.076302,0.134890,0.418994,0.374814,3538,0.622951,0.603096,0.418994,0.374814
7,optuna_object_matrix_0148,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,alternative,alternative_empHQ_fixed2,alternative_empHQ_fixed2,5.0,0.01,0.50,15,2,3,40,NaN,random,NaN,alm5pea2,peanut,almond,4014,102,66,1372,2474,0.607143,0.643266,0.625204,0.641754,0.069199,0.124239,0.392857,0.356734,4014,0.641754,0.625204,0.392857,0.356734
8,optuna_object_matrix_0184,04B2_optuna_binary_pareto__3way_calibrated__04...,object_matrix,obj

## 4B. Two-way projection tables and confidence

In [11]:
# 2-way confidence and confusion tables
# ---------------------------------------------------------------------
# These are complete binary confusion tables. Binary confidence remains stored
# in the prediction tables, but is not aggregated by binary_confusion_table.
# Confidence is only a diagnostic:
# - object level: distance from the object_threshold in target-pixel-ratio space
# - pixel level: distance from the SIMCA rule limit when rule_statistic/rule_limit exist

mixture_objects_df = add_binary_object_confidence(
    mixture_objects_df,
    target_class=TARGET_CLASS,
)

mixture_pixels_df = add_binary_pixel_confidence(
    mixture_pixels_df,
    score_col="rule_statistic",
    threshold_col="rule_limit",
    accepted_when_below=True,
)

object_true_col = true_col(TARGET_CLASS, "object")
object_pred_col = predicted_col(TARGET_CLASS, "object")
pixel_true_col = true_col(TARGET_CLASS, "pixel")
pixel_pred_col = predicted_col(TARGET_CLASS, "pixel")

TWO_WAY_GROUP_COLS = [
    "selected_config_id",
    "matrix_family",
    "candidate_source",
]

mixture_object_2way_confusion_df = (
    binary_confusion_table(
        mixture_objects_df,
        true_col=object_true_col,
        pred_col=object_pred_col,
        group_cols=TWO_WAY_GROUP_COLS,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )
)

mixture_pixel_2way_confusion_df = (
    binary_confusion_table(
        mixture_pixels_df,
        true_col=pixel_true_col,
        pred_col=pixel_pred_col,
        group_cols=TWO_WAY_GROUP_COLS,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )
)

save_parquet_if_nonempty(
    mixture_object_2way_confusion_df,
    MIXTURE_OBJECT_2WAY_CONFUSION_PATH,
)

save_parquet_if_nonempty(
    mixture_pixel_2way_confusion_df,
    MIXTURE_PIXEL_2WAY_CONFUSION_PATH,
)

print("Object 2-way confusion:", mixture_object_2way_confusion_df.shape)
print("Pixel 2-way confusion:", mixture_pixel_2way_confusion_df.shape)
print("Saved:")
print(" -", MIXTURE_OBJECT_2WAY_CONFUSION_PATH)
print(" -", MIXTURE_PIXEL_2WAY_CONFUSION_PATH)

display(mixture_object_2way_confusion_df.head(20))


Object 2-way confusion: (64, 12)
Pixel 2-way confusion: (64, 12)
Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_2way_confusion.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_pixel_2way_confusion.parquet


,selected_config_id,matrix_family,candidate_source,true_label_2way,predicted_label_2way,n,n_true_label,n_group,row_rate,global_rate,mean_confidence,median_confidence
0,04A_object_matrix_0002,object_matrix,04B_grid_robustness,almond,almond,434,576,722,0.753472,0.601108,0.391306,0.361208
1,04A_object_matrix_0002,object_matrix,04B_grid_robustness,almond,peanut,142,576,722,0.246528,0.196676,0.307690,0.292482
2,04A_object_matrix_0002,object_matrix,04B_grid_robustness,peanut,almond,35,146,722,0.239726,0.048476,0.140308,0.104762
3,04A_object_matrix_0002,object_matrix,04B_grid_robustness,peanut,peanut,111,146,722,0.760274,0.153740,0.379481,0.365079
4,04A_object_matrix_0003,object_matrix,04B_grid_robustness,almond,almond,477,576,722,0.828125,0.660665,0.396441,0.370143
5,04A_object_matrix_0003,object_matrix,04B_grid_robustness,almond,peanut,99,576,722,0.171875,0.137119,0.293151,0.262136
6,04A_object_matrix_0003,object_matrix,04B_grid_robustness,peanut,almond,58,146,722,0.397260,0.080332,0.130247,0.083333
7,04A_object_matrix_0003,object_matrix,04B_grid_robustness,peanut,peanut,88,146,722,0.602740,0.121884,0.343858,0.353095
8,04A_object_matrix_0004,object_matrix,04B_grid_robustness,almond,almond,449,576,722,0.779514,0.621884,0.393064,0.371921
9,04A_object_matrix_0004,object_matrix,04B_grid_robustness,almond,peanut,127,576,722,0.220486,0.175900,0.316829,0.292035


## 5. Optional 3-way object decision on mixture images

In [12]:
# ---------------------------------------------------------------------
# Apply fixed 3-way object decisions on mixture images
# ---------------------------------------------------------------------
# No threshold calibration is performed here.
# Thresholds come from frozen_reference_configs.parquet.

if APPLY_FIXED_THREE_WAY:
    mixture_three_way_metrics_df, mixture_objects_3way_df = evaluate_three_way_by_config(
        object_df=mixture_objects_df,
        thresholds_df=reference_configs_df,
        config_id_col="selected_config_id",
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    mixture_three_way_by_image_df, _ = evaluate_three_way_by_config(
        object_df=mixture_objects_df,
        thresholds_df=reference_configs_df,
        config_id_col="selected_config_id",
        extra_group_cols=["source_image"],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )
    if "n_target" in mixture_three_way_by_image_df.columns:
        mixture_three_way_by_image_df[
            "n_true_target_objects"
        ] = (
            mixture_three_way_by_image_df["n_target"]
            .fillna(0)
            .astype(int)
        )

    mixture_objects_3way_df = add_three_way_confidence(
        mixture_objects_3way_df,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        uncertain_label=UNCERTAIN_LABEL_USED,
        decision_col="decision_3way",
    )

    mixture_objects_3way_df = merge_config_metadata(
        mixture_objects_3way_df,
        reference_configs_df,
        id_col="selected_config_id",
        columns=REFERENCE_METADATA_COLS,
    )

else:
    mixture_three_way_metrics_df = pd.DataFrame()
    mixture_three_way_by_image_df = pd.DataFrame()
    mixture_objects_3way_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_three_way_metrics_df,
    MIXTURE_THREE_WAY_METRICS_PATH,
)

save_parquet_if_nonempty(
    mixture_three_way_by_image_df,
    MIXTURE_THREE_WAY_BY_IMAGE_PATH,
)

save_parquet_if_nonempty(
    mixture_objects_3way_df,
    MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH,
)

print("Fixed 3-way metrics:", mixture_three_way_metrics_df.shape)
print("Fixed 3-way by image:", mixture_three_way_by_image_df.shape)
print("Mixture objects with fixed 3-way decisions:", mixture_objects_3way_df.shape)

display(
    mixture_three_way_metrics_df[
        [
            col for col in [
                "selected_config_id",
                "n",
                "n_target",
                "n_non_target",
                "target_miss_rate",
                "screening_sensitivity",
                "non_target_false_accept_rate",
                "non_target_auto_reject_rate",
                "uncertain_rate",
                "coverage_rate",
                "decided_balanced_accuracy",
            ]
            if col in mixture_three_way_metrics_df.columns
        ]
    ].head(30)
)

Fixed 3-way metrics: (16, 21)
Fixed 3-way by image: (320, 23)
Mixture objects with fixed 3-way decisions: (11552, 80)


,selected_config_id,n,n_target,n_non_target,target_miss_rate,screening_sensitivity,non_target_false_accept_rate,non_target_auto_reject_rate,uncertain_rate,coverage_rate,decided_balanced_accuracy
0,04A_object_matrix_0002,722,146,576,0.061644,0.938356,0.000000,0.532986,0.560942,0.439058,0.550000
1,04A_object_matrix_0003,722,146,576,0.061644,0.938356,0.000000,0.532986,0.560942,0.439058,0.550000
2,04A_object_matrix_0004,722,146,576,0.089041,0.910959,0.000000,0.539931,0.551247,0.448753,0.500000
3,04A_object_matrix_0007,722,146,576,0.006849,0.993151,0.072917,0.267361,0.619114,0.380886,0.886528
4,04A_pixel_matrix_0001,722,146,576,0.000000,1.000000,0.147569,0.244792,0.547091,0.452909,0.811947
5,04A_pixel_matrix_0002,722,146,576,0.000000,1.000000,0.078125,0.505208,0.400277,0.599723,0.933036
6,04A_pixel_matrix_0004,722,146,576,0.000000,1.000000,0.147569,0.244792,0.547091,0.452909,0.811947
7,04A_pixel_matrix_0005,722,146,576,0.000000,1.000000,0.119792,0.675347,0.186981,0.813019,0.924672
8,optuna_object_matrix_0081,722,146,576,0.047945,0.952055,0.029514,0.515625,0.501385,0.498615,0.896843
9,optuna_object_matrix_0148,722,146,576,0.047945,0.952055,0.029514,0.479167,0.537396,0.462604,0.885624


In [13]:
# ---------------------------------------------------------------------
# Assign objectwise 3-way decisions to pixels
# ---------------------------------------------------------------------
# This is a spatial view of the OBJECT decision, not an independent pixel-level
# three-way classifier.

if APPLY_FIXED_THREE_WAY and len(mixture_objects_3way_df) > 0:
    mixture_pixels_3way_df = mixture_pixels_df.copy()

    columns_to_transfer = [
        "decision_3way",
        "three_way_lower_threshold",
        "three_way_upper_threshold",
        "three_way_confidence",
        "three_way_margin",
    ]

    for column in columns_to_transfer:
        if column not in mixture_objects_3way_df.columns:
            continue

        mixture_pixels_3way_df = assign_object_decisions_to_pixels(
            pixel_df=mixture_pixels_3way_df,
            object_df=mixture_objects_3way_df,
            decision_col=column,
            output_col=column,
        )
else:
    mixture_pixels_3way_df = pd.DataFrame()

if SAVE_MIXTURE_PIXEL_TABLES:
    save_parquet_if_nonempty(
        mixture_pixels_3way_df,
        MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH,
    )

print("Mixture pixels with objectwise 3-way decision:", mixture_pixels_3way_df.shape)
print("Full 3-way pixel table saved:", bool(SAVE_MIXTURE_PIXEL_TABLES))


Mixture pixels with objectwise 3-way decision: (1020352, 102)
Full 3-way pixel table saved: False


In [14]:
# ---------------------------------------------------------------------
# 3-way confusion tables with confidence
# ---------------------------------------------------------------------

object_true_col = true_col(TARGET_CLASS, "object")
pixel_true_col = true_col(TARGET_CLASS, "pixel")

if len(mixture_objects_3way_df) > 0:
    mixture_object_3way_confusion_df = three_way_confusion_table(
        df=mixture_objects_3way_df,
        true_col=object_true_col,
        decision_col="decision_3way",
        confidence_col="three_way_confidence",
        group_cols=[
            "selected_config_id",
            "matrix_family",
            "candidate_source",
        ],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        uncertain_label=UNCERTAIN_LABEL_USED,
    )
else:
    mixture_object_3way_confusion_df = pd.DataFrame()

if len(mixture_pixels_3way_df) > 0:
    mixture_pixel_3way_confusion_df = three_way_confusion_table(
        df=mixture_pixels_3way_df,
        true_col=pixel_true_col,
        decision_col="decision_3way",
        confidence_col="three_way_confidence",
        group_cols=[
            "selected_config_id",
            "matrix_family",
            "candidate_source",
        ],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        uncertain_label=UNCERTAIN_LABEL_USED,
    )
else:
    mixture_pixel_3way_confusion_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_object_3way_confusion_df,
    MIXTURE_OBJECT_3WAY_CONFUSION_PATH,
)
save_parquet_if_nonempty(
    mixture_pixel_3way_confusion_df,
    MIXTURE_PIXEL_3WAY_CONFUSION_PATH,
)

print("Object 3-way confusion:", mixture_object_3way_confusion_df.shape)
print("Pixel view of objectwise 3-way confusion:", mixture_pixel_3way_confusion_df.shape)

display(mixture_object_3way_confusion_df.head(20))


Object 3-way confusion: (96, 12)
Pixel view of objectwise 3-way confusion: (96, 12)


,selected_config_id,matrix_family,candidate_source,true_label_3way,decision_3way,n,n_true_label,n_group,row_rate,global_rate,mean_confidence,median_confidence
0,04A_object_matrix_0002,object_matrix,04B_grid_robustness,almond,almond,307,576,722,0.532986,0.425208,0.377438,0.336898
1,04A_object_matrix_0002,object_matrix,04B_grid_robustness,almond,uncertain,269,576,722,0.467014,0.372576,0.554799,0.583333
2,04A_object_matrix_0002,object_matrix,04B_grid_robustness,almond,peanut,0,576,722,0.000000,0.000000,NaN,NaN
3,04A_object_matrix_0002,object_matrix,04B_grid_robustness,peanut,almond,9,146,722,0.061644,0.012465,0.124015,0.124579
4,04A_object_matrix_0002,object_matrix,04B_grid_robustness,peanut,uncertain,136,146,722,0.931507,0.188366,0.610436,0.654884
5,04A_object_matrix_0002,object_matrix,04B_grid_robustness,peanut,peanut,1,146,722,0.006849,0.001385,0.411765,0.411765
6,04A_object_matrix_0003,object_matrix,04B_grid_robustness,almond,almond,307,576,722,0.532986,0.425208,0.377438,0.336898
7,04A_object_matrix_0003,object_matrix,04B_grid_robustness,almond,uncertain,269,576,722,0.467014,0.372576,0.554799,0.583333
8,04A_object_matrix_0003,object_matrix,04B_grid_robustness,almond,peanut,0,576,722,0.000000,0.000000,NaN,NaN
9,04A_object_matrix_0003,object_matrix,04B_grid_robustness,peanut,almond,9,146,722,0.061644,0.012465,0.124015,0.124579


In [15]:
# ---------------------------------------------------------------------
# SIMCA Q residuals / Hotelling T² diagnostic tables
# ---------------------------------------------------------------------
# The notebook does not plot these diagnostics. They can be consumed by the
# external reporting script. Full pixel diagnostics are saved only when
# SAVE_MIXTURE_PIXEL_TABLES=True.

SIMCA_DIAG_GROUP_KEYS = [
    "selected_config_id",
    "source_image",
    "object_id",
]

pixel_diag_cols = [
    "H",
    "Q",
    "H_norm_limit",
    "Q_norm_limit",
    "rule_statistic",
    "rule_limit",
]
available_pixel_diag_cols = [
    col for col in pixel_diag_cols
    if col in mixture_pixels_3way_df.columns
]

if len(mixture_pixels_3way_df) > 0 and available_pixel_diag_cols:
    object_simca_diag_df = (
        mixture_pixels_3way_df
        .groupby(SIMCA_DIAG_GROUP_KEYS, dropna=False)
        .agg(
            **{
                f"{col}_mean": (col, "mean")
                for col in available_pixel_diag_cols
            },
            **{
                f"{col}_median": (col, "median")
                for col in available_pixel_diag_cols
            },
            n_pixels_diag=("object_id", "size"),
        )
        .reset_index()
    )

    diagnostic_output_cols = [
        f"{col}_{stat}"
        for col in available_pixel_diag_cols
        for stat in ("mean", "median")
    ] + ["n_pixels_diag"]

    mixture_objects_3way_df = mixture_objects_3way_df.drop(
        columns=[
            col for col in diagnostic_output_cols
            if col in mixture_objects_3way_df.columns
        ],
        errors="ignore",
    )

    mixture_objects_3way_df = mixture_objects_3way_df.merge(
        object_simca_diag_df,
        on=SIMCA_DIAG_GROUP_KEYS,
        how="left",
        validate="one_to_one",
    )
else:
    object_simca_diag_df = pd.DataFrame()

pixel_keep_cols = [
    col for col in [
        "selected_config_id",
        "source_image",
        "object_id",
        "row",
        "col",
        "predicted_label_pixel",
        "decision_3way",
        "binary_confidence",
        "three_way_confidence",
        "H",
        "Q",
        "H_norm_limit",
        "Q_norm_limit",
        "rule_statistic",
        "rule_limit",
        "pixel_error_case",
    ]
    if col in mixture_pixels_3way_df.columns
]

object_keep_cols = [
    col for col in [
        "selected_config_id",
        "source_image",
        "object_id",
        "true_label_object",
        "predicted_label_object",
        "decision_3way",
        "binary_confidence",
        "three_way_confidence",
        "three_way_margin",
        "H_mean",
        "Q_mean",
        "H_norm_limit_mean",
        "Q_norm_limit_mean",
        "rule_statistic_mean",
        "rule_limit_mean",
        "object_error_case",
    ]
    if col in mixture_objects_3way_df.columns
]

mixture_object_simca_diagnostics_df = (
    mixture_objects_3way_df[object_keep_cols].copy()
    if object_keep_cols
    else pd.DataFrame()
)

mixture_pixel_simca_diagnostics_df = (
    mixture_pixels_3way_df[pixel_keep_cols].copy()
    if pixel_keep_cols
    else pd.DataFrame()
)

save_parquet_if_nonempty(
    mixture_object_simca_diagnostics_df,
    MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH,
)

if SAVE_MIXTURE_PIXEL_TABLES:
    save_parquet_if_nonempty(
        mixture_pixel_simca_diagnostics_df,
        MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH,
    )

print("Object SIMCA diagnostics:", mixture_object_simca_diagnostics_df.shape)
print("Pixel SIMCA diagnostics in memory:", mixture_pixel_simca_diagnostics_df.shape)
print("Full pixel diagnostic table saved:", bool(SAVE_MIXTURE_PIXEL_TABLES))


Object SIMCA diagnostics: (11552, 16)
Pixel SIMCA diagnostics in memory: (1020352, 16)
Full pixel diagnostic table saved: False


## 6. Save object predictions and optional minimal pixel table

In [16]:
# Binary and 3-way object tables are kept separate to avoid duplicated exports.
save_parquet(mixture_objects_df, MIXTURE_OBJECT_PREDICTIONS_PATH)
save_parquet_if_nonempty(
    mixture_objects_3way_df,
    MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH,
)

print("Saved object predictions:")
print(" -", MIXTURE_OBJECT_PREDICTIONS_PATH)
if len(mixture_objects_3way_df) > 0:
    print(" -", MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH)

if SAVE_MIXTURE_PIXEL_TABLES:
    pixel_source_df = (
        mixture_pixels_3way_df
        if len(mixture_pixels_3way_df) > 0
        else mixture_pixels_df
    )

    minimal_pixel_cols = [
        col for col in [
            "selected_config_id",
            "source_image",
            "object_id",
            "row",
            "col",
            true_col(TARGET_CLASS, "pixel"),
            predicted_col(TARGET_CLASS, "pixel"),
            "truth_available",
            "predicted_label_pixel",
            "pixel_error_case",
            "decision_3way",
            "binary_confidence",
            "three_way_confidence",
            "H_norm_limit",
            "Q_norm_limit",
            "rule_statistic",
            "rule_limit",
        ]
        if col in pixel_source_df.columns
    ]

    save_parquet(
        pixel_source_df[minimal_pixel_cols].copy(),
        MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH,
    )
    print(" -", MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH)


Saved object predictions:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_predictions.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_predictions_3way.parquet


In [17]:
THREE_WAY_SUMMARY_COLS = [
    "selected_config_id",
    "n",
    "n_target",
    "n_non_target",
    "target_miss_rate",
    "screening_sensitivity",
    "non_target_false_accept_rate",
    "non_target_auto_reject_rate",
    "uncertain_rate",
    "coverage_rate",
    "decided_balanced_accuracy",
]

THREE_WAY_SUMMARY_COLS = [
    col for col in THREE_WAY_SUMMARY_COLS
    if col in mixture_three_way_metrics_df.columns
]

if THREE_WAY_SUMMARY_COLS:
    three_way_prefixed = (
        mixture_three_way_metrics_df[THREE_WAY_SUMMARY_COLS]
        .rename(
            columns={
                "n": "object_3way_n",
                "n_target": "object_3way_n_target",
                "n_non_target": "object_3way_n_non_target",
                "target_miss_rate": "object_3way_target_miss_rate",
                "screening_sensitivity": "object_3way_screening_sensitivity",
                "non_target_false_accept_rate": "object_3way_non_target_false_accept_rate",
                "non_target_auto_reject_rate": "object_3way_non_target_auto_reject_rate",
                "uncertain_rate": "object_3way_uncertain_rate",
                "coverage_rate": "object_3way_coverage_rate",
                "decided_balanced_accuracy": "object_3way_decided_balanced_accuracy",
            }
        )
    )

    # Make the cell rerunnable by dropping previous 3-way summary columns.
    three_way_cols_to_drop = [
        col for col in three_way_prefixed.columns
        if col != "selected_config_id" and col in mixture_model_summary_df.columns
    ]
    if three_way_cols_to_drop:
        mixture_model_summary_df = mixture_model_summary_df.drop(columns=three_way_cols_to_drop)

    mixture_model_summary_df = mixture_model_summary_df.merge(
        three_way_prefixed,
        on="selected_config_id",
        how="left",
    )

sort_cols = [
    "frozen_reference_rank",
    "matrix_family",
    "candidate_source",
    "selected_config_id",
]
sort_cols = [col for col in sort_cols if col in mixture_model_summary_df.columns]

mixture_model_summary_df = (
    mixture_model_summary_df
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

save_parquet(mixture_model_summary_df, MIXTURE_MODEL_SUMMARY_PATH)

print("Updated mixture model summary with fixed 3-way metrics:", mixture_model_summary_df.shape)
print("Saved:", MIXTURE_MODEL_SUMMARY_PATH)

display(
    mixture_model_summary_df[
        [
            col for col in [
                "selected_config_id",
                "frozen_reference_rank",
                "candidate_source",
                "matrix_family",
                "training_matrix_id",
                "preprocessing",
                "selected_rule_name",
                "object_fn_rate",
                "object_fp_rate",
                "pixel_fn_rate",
                "pixel_fp_rate",
                "object_3way_target_miss_rate",
                "object_3way_non_target_false_accept_rate",
                "object_3way_uncertain_rate",
                "object_3way_coverage_rate",
            ]
            if col in mixture_model_summary_df.columns
        ]
    ].head(80)
)

Updated mixture model summary with fixed 3-way metrics: (16, 58)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_model_summary.parquet


,selected_config_id,matrix_family,training_matrix_id,preprocessing,selected_rule_name,object_fn_rate,object_fp_rate,pixel_fn_rate,pixel_fp_rate,object_3way_target_miss_rate,object_3way_non_target_false_accept_rate,object_3way_uncertain_rate,object_3way_coverage_rate
0,04A_object_matrix_0002,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,0.239726,0.246528,0.227846,0.505331,0.061644,0.000000,0.560942,0.439058
1,04A_object_matrix_0003,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,0.397260,0.171875,0.227846,0.505331,0.061644,0.000000,0.560942,0.439058
2,04A_object_matrix_0004,object_matrix,object_median,absorbance_sg_d2,simple_emp_cv,0.328767,0.220486,0.253151,0.494482,0.089041,0.000000,0.551247,0.448753
3,04A_object_matrix_0007,object_matrix,object_median,raw,data_driven_emp_cv,0.219178,0.201389,0.192069,0.479292,0.006849,0.072917,0.619114,0.380886
4,optuna_object_matrix_0081,object_matrix,object_median,sg_smooth,data_driven_emp_cv,0.047945,0.484375,0.169188,0.466983,0.047945,0.029514,0.501385,0.498615
5,optuna_object_matrix_0148,object_matrix,object_median,sg_smooth,alternative_empHQ_fixed2,0.047945,0.520833,0.189936,0.478993,0.047945,0.029514,0.537396,0.462604
6,optuna_object_matrix_0149,object_matrix,object_median,sg_smooth,alternative_chi2_emp_cv,0.047945,0.491319,0.166861,0.470481,0.047945,0.034722,0.501385,0.498615
7,optuna_object_matrix_0184,object_matrix,object_median,absorbance_sg_smooth,simple_emp_cv,0.082192,0.317708,0.286019,0.354877,0.000000,0.095486,0.490305,0.509695
8,04A_pixel_matrix_0001,pixel_matrix,balanced_pixel_random_m40,absorbance_sg_smooth,simple_chi2,0.006849,0.552083,0.042079,0.694508,0.000000,0.147569,0.547091,0.452909
9,04A_pixel_matrix_0002,pixel_matrix,balanced_pixel_center_m40,absorbance_sg_smooth,simple_emp_cv,0.013699,0.333333,0.081152,0.508960,0.000000,0.078125,0.400277,0.599723


## 7. Border/core diagnostics on mixture images

In [18]:
if RUN_BORDER_DIAGNOSTIC:
    mixture_border_diagnostic_df = summarize_border_diagnostics_by_config(
        pixel_df=mixture_pixels_df,
        object_db=object_db,
        target_class=TARGET_CLASS,
        border_widths=BORDER_DIAGNOSTIC_WIDTHS,
        config_cols=[
            "selected_config_id",
            "matrix_family",
            "training_matrix_id",
            "matrix_method",
            "preprocessing",
            "selected_rule_name",
            "n_components",
            "alpha",
            "object_threshold",
        ],
    )
else:
    mixture_border_diagnostic_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_border_diagnostic_df,
    MIXTURE_BORDER_DIAGNOSTIC_PATH,
)

print("Mixture border diagnostic:", mixture_border_diagnostic_df.shape)

display(mixture_border_diagnostic_df.head(80))

Mixture border diagnostic: (96, 21)


,selected_config_id,matrix_family,training_matrix_id,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,border_width,zone,n_pixels,tp,tn,fp,fn,n_errors,error_rate,fp_rate,fn_rate,pixel_accuracy
0,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,1,border,16478,1813,7954,5598,1113,6711,0.407270,0.413076,0.380383,0.592730
1,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,1,core,47294,6151,18490,21416,1237,22653,0.478983,0.536661,0.167434,0.521017
2,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,2,border,32654,4080,13844,13061,1669,14730,0.451093,0.485449,0.290311,0.548907
3,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,2,core,31118,3884,12600,13953,681,14634,0.470274,0.525477,0.149179,0.529726
4,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,3,border,49177,6328,20065,20682,2102,22784,0.463306,0.507571,0.249348,0.536694
5,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,3,core,14595,1636,6379,6332,248,6580,0.450839,0.498151,0.131635,0.549161
6,04A_object_matrix_0003,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.75,1,border,16478,1813,7954,5598,1113,6711,0.407270,0.413076,0.380383,0.592730
7,04A_object_matrix_0003,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.75,1,core,47294,6151,18490,21416,1237,22653,0.478983,0.536661,0.167434,0.521017
8,04A_object_matrix_0003,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.75,2,border,32654,4080,13844,13061,1669,14730,0.451093,0.485449,0.290311,0.548907
9,04A_object_matrix_0003,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.75,2,core,31118,3884,12600,13953,681,14634,0.470274,0.525477,0.149179,0.529726


## 8. Sensitivity to mixture-truth dilation radius

In [19]:
def evaluate_mixture_truth_dilation_sensitivity(
    pixel_df: pd.DataFrame,
    config_df: pd.DataFrame,
    image_db: dict,
    object_db: dict,
    target_class: str,
    non_target_label: str,
    dilation_radii,
) -> pd.DataFrame:
    """
    Recompute mixture truth maps with several dilation radii.

    Predictions are kept fixed. Only the approximate truth labels are changed.
    This helps determine whether model ranking depends strongly on the
    position-reference truth construction.
    """
    rows = []

    config_lookup = (
        config_df
        .drop_duplicates("selected_config_id")
        .set_index("selected_config_id")
        .to_dict(orient="index")
    )

    for dilation_radius in dilation_radii:
        pixel_truth_df = add_pixel_truth_labels(
            pixel_df=pixel_df,
            image_db=image_db,
            object_db=object_db,
            target_class=target_class,
            dilation_radius=int(dilation_radius),
        )

        for config_id, group in pixel_truth_df.groupby("selected_config_id", dropna=False):
            config_id = str(config_id)
            cfg = config_lookup.get(config_id, {})

            object_threshold = float(cfg.get("object_threshold", 0.75))

            threshold_df, _object_tables = object_threshold_grid(
                pixel_df=group,
                object_db=object_db,
                target_class=target_class,
                non_target_label=non_target_label,
                thresholds=[object_threshold],
            )

            if threshold_df is None or len(threshold_df) == 0:
                continue

            row = threshold_df.iloc[0].to_dict()
            row["selected_config_id"] = config_id
            row["truth_dilation_radius"] = int(dilation_radius)

            for col in [
                "matrix_family",
                "training_matrix_id",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
            ]:
                if col in cfg:
                    row[col] = cfg[col]

            rows.append(row)

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["selected_config_id", "truth_dilation_radius"],
            ascending=True,
        )
        .reset_index(drop=True)
    )


if RUN_TRUTH_DILATION_SENSITIVITY:
    mixture_truth_dilation_sensitivity_df = evaluate_mixture_truth_dilation_sensitivity(
        pixel_df=mixture_pixels_df,
        config_df=reference_configs_df,
        image_db=image_db,
        object_db=object_db,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        dilation_radii=TRUTH_DILATION_RADII,
    )
else:
    mixture_truth_dilation_sensitivity_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_truth_dilation_sensitivity_df,
    MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH,
)

print("Mixture truth dilation sensitivity:", mixture_truth_dilation_sensitivity_df.shape)

display(mixture_truth_dilation_sensitivity_df.head(80))

C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (s

Mixture truth dilation sensitivity: (96, 25)


,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,object_threshold,selected_config_id,truth_dilation_radius,matrix_family,training_matrix_id,matrix_method,preprocessing,selected_rule_name,n_components,alpha
0,peanut,almond,722,111,35,142,434,0.760274,0.753472,0.756873,0.754848,0.438735,0.556391,0.239726,0.246528,0.70,04A_object_matrix_0002,0,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
1,peanut,almond,722,111,35,142,434,0.760274,0.753472,0.756873,0.754848,0.438735,0.556391,0.239726,0.246528,0.70,04A_object_matrix_0002,1,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
2,peanut,almond,722,111,35,142,434,0.760274,0.753472,0.756873,0.754848,0.438735,0.556391,0.239726,0.246528,0.70,04A_object_matrix_0002,2,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
3,peanut,almond,722,111,35,142,434,0.760274,0.753472,0.756873,0.754848,0.438735,0.556391,0.239726,0.246528,0.70,04A_object_matrix_0002,3,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
4,peanut,almond,722,111,35,142,434,0.760274,0.753472,0.756873,0.754848,0.438735,0.556391,0.239726,0.246528,0.70,04A_object_matrix_0002,4,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
5,peanut,almond,722,111,35,142,434,0.760274,0.753472,0.756873,0.754848,0.438735,0.556391,0.239726,0.246528,0.70,04A_object_matrix_0002,5,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
6,peanut,almond,722,88,58,99,477,0.602740,0.828125,0.715432,0.782548,0.470588,0.528529,0.397260,0.171875,0.75,04A_object_matrix_0003,0,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
7,peanut,almond,722,88,58,99,477,0.602740,0.828125,0.715432,0.782548,0.470588,0.528529,0.397260,0.171875,0.75,04A_object_matrix_0003,1,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
8,peanut,almond,722,88,58,99,477,0.602740,0.828125,0.715432,0.782548,0.470588,0.528529,0.397260,0.171875,0.75,04A_object_matrix_0003,2,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01
9,peanut,almond,722,88,58,99,477,0.602740,0.828125,0.715432,0.782548,0.470588,0.528529,0.397260,0.171875,0.75,04A_object_matrix_0003,3,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01


In [20]:
if len(mixture_truth_dilation_sensitivity_df) > 0:
    truth_sensitivity_summary_df = (
        mixture_truth_dilation_sensitivity_df
        .groupby("selected_config_id", dropna=False)
        .agg(
            n_radii=("truth_dilation_radius", "nunique"),
            min_fn_rate=("fn_rate", "min"),
            max_fn_rate=("fn_rate", "max"),
            range_fn_rate=(
                "fn_rate",
                lambda s: float(
                    pd.to_numeric(s, errors="coerce").max()
                    - pd.to_numeric(s, errors="coerce").min()
                ),
            ),
            min_fp_rate=("fp_rate", "min"),
            max_fp_rate=("fp_rate", "max"),
            range_fp_rate=(
                "fp_rate",
                lambda s: float(
                    pd.to_numeric(s, errors="coerce").max()
                    - pd.to_numeric(s, errors="coerce").min()
                ),
            ),
            min_balanced_accuracy=("balanced_accuracy", "min"),
            max_balanced_accuracy=("balanced_accuracy", "max"),
            range_balanced_accuracy=(
                "balanced_accuracy",
                lambda s: float(
                    pd.to_numeric(s, errors="coerce").max()
                    - pd.to_numeric(s, errors="coerce").min()
                ),
            ),
        )
        .reset_index()
        .sort_values(["range_fn_rate", "range_fp_rate"], ascending=False)
    )

    print("Truth-sensitivity summary by model:")
    display(truth_sensitivity_summary_df.head(20))
else:
    truth_sensitivity_summary_df = pd.DataFrame()


Truth-sensitivity summary by model:


,selected_config_id,n_radii,min_fn_rate,max_fn_rate,range_fn_rate,min_fp_rate,max_fp_rate,range_fp_rate,min_balanced_accuracy,max_balanced_accuracy,range_balanced_accuracy
0,04A_object_matrix_0002,6,0.239726,0.239726,0.0,0.246528,0.246528,0.0,0.756873,0.756873,0.0
1,04A_object_matrix_0003,6,0.397260,0.397260,0.0,0.171875,0.171875,0.0,0.715432,0.715432,0.0
2,04A_object_matrix_0004,6,0.328767,0.328767,0.0,0.220486,0.220486,0.0,0.725373,0.725373,0.0
3,04A_object_matrix_0007,6,0.219178,0.219178,0.0,0.201389,0.201389,0.0,0.789717,0.789717,0.0
4,04A_pixel_matrix_0001,6,0.006849,0.006849,0.0,0.552083,0.552083,0.0,0.720534,0.720534,0.0
5,04A_pixel_matrix_0002,6,0.013699,0.013699,0.0,0.333333,0.333333,0.0,0.826484,0.826484,0.0
6,04A_pixel_matrix_0004,6,0.027397,0.027397,0.0,0.454861,0.454861,0.0,0.758871,0.758871,0.0
7,04A_pixel_matrix_0005,6,0.089041,0.089041,0.0,0.168403,0.168403,0.0,0.871278,0.871278,0.0
8,optuna_object_matrix_0081,6,0.047945,0.047945,0.0,0.484375,0.484375,0.0,0.733840,0.733840,0.0
9,optuna_object_matrix_0148,6,0.047945,0.047945,0.0,0.520833,0.520833,0.0,0.715611,0.715611,0.0


## 9. Parameter tendencies among frozen reference models

In [21]:
mixture_parameter_tendencies_df = summarize_parameter_tendencies(
    mixture_model_summary_df.rename(
        columns={
            "object_fn_rate": "fn_rate",
            "object_fp_rate": "fp_rate",
            "object_balanced_accuracy": "balanced_accuracy",
        }
    ),
    top_fraction=1.0,
    min_top_n=len(mixture_model_summary_df),
)

save_parquet_if_nonempty(
    mixture_parameter_tendencies_df,
    MIXTURE_PARAMETER_TENDENCIES_PATH,
)

print("Mixture parameter tendencies:", mixture_parameter_tendencies_df.shape)

display(mixture_parameter_tendencies_df.head(80))

Mixture parameter tendencies: (69, 6)


,parameter,value,count,rate_in_top_models,matrix_family,n_top_models
0,alpha,0.009999999776482582,8,1.000,object_matrix,8
1,balanced_pixel_strategy,random,4,0.500,object_matrix,8
2,balanced_pixel_strategy,not_applicable,4,0.500,object_matrix,8
3,m,40,8,1.000,object_matrix,8
4,matrix_method,object_median,8,1.000,object_matrix,8
5,n_components,NaN,4,0.500,object_matrix,8
6,n_components,5.0,3,0.375,object_matrix,8
7,n_components,8.0,1,0.125,object_matrix,8
8,object_threshold,0.5,4,0.500,object_matrix,8
9,object_threshold,0.75,2,0.250,object_matrix,8


## 10. Essential visual diagnostics

The notebook intentionally shows only one reference configuration and one difficult
mixture image. At most four figures are produced:

1. object-level binary confusion matrix;
2. object-level 3-way confusion matrix;
3. one four-panel spatial diagnostic;
4. sensitivity to the truth-map dilation radius.

The external reporting script should be used for exhaustive plots over all
configurations and images.


In [22]:
# ---------------------------------------------------------------------
# Select one frozen reference configuration
# ---------------------------------------------------------------------
if DIAGNOSTIC_CONFIG_ID is not None:
    # An explicit identifier takes priority, even when the model did not pass
    # the optional pure-test guardrail.
    diagnostic_config_df = reference_configs_df[
        reference_configs_df["selected_config_id"]
        .astype(str)
        .eq(str(DIAGNOSTIC_CONFIG_ID))
    ].copy()

    if diagnostic_config_df.empty:
        raise KeyError(
            f"DIAGNOSTIC_CONFIG_ID={DIAGNOSTIC_CONFIG_ID!r} was not found "
            "among the frozen reference configurations."
        )
else:
    diagnostic_pool_df = reference_configs_df.copy()

    if "passes_pure_test_guardrail" in diagnostic_pool_df.columns:
        guardrail_raw = diagnostic_pool_df["passes_pure_test_guardrail"]
        guardrail_mask = (
            guardrail_raw.eq(True)
            | guardrail_raw.astype(str).str.strip().str.lower().isin(
                {"true", "1", "yes", "y"}
            )
        )
        if guardrail_mask.any():
            diagnostic_pool_df = diagnostic_pool_df[guardrail_mask].copy()

    diagnostic_sort_cols = [
        col for col in ["frozen_reference_rank", "selected_config_id"]
        if col in diagnostic_pool_df.columns
    ]
    diagnostic_config_df = (
        diagnostic_pool_df
        .sort_values(diagnostic_sort_cols)
        .head(1)
        .copy()
    )

if diagnostic_config_df.empty:
    raise RuntimeError("No frozen reference configuration available for diagnostics.")

diagnostic_config_id = str(
    diagnostic_config_df.iloc[0]["selected_config_id"]
)

# ---------------------------------------------------------------------
# Select one difficult image for that configuration
# ---------------------------------------------------------------------
if APPLY_FIXED_THREE_WAY and len(mixture_three_way_by_image_df) > 0:
    diagnostic_image_df = choose_images_for_config_3way(
        mixture_three_way_by_image_df,
        config_id=diagnostic_config_id,
        config_col="selected_config_id",
        image_col="source_image",
        uncertain_col="uncertain_rate",
        miss_col="target_miss_rate",
        false_accept_col="non_target_false_accept_rate",
        target_count_col="n_true_target_objects",
        n_images=N_DIAGNOSTIC_IMAGES,
        max_single_target_images=1,
    )
else:
    diagnostic_image_df = choose_images_for_config(
        mixture_object_errors_by_image_df,
        config_id=diagnostic_config_id,
        config_col="selected_config_id",
        image_col="source_image",
        fn_col="fn_rate",
        fp_col="fp_rate",
        target_count_col="n_true_target_objects",
        n_images=N_DIAGNOSTIC_IMAGES,
        max_single_target_images=1,
    )

if diagnostic_image_df.empty:
    raise RuntimeError(
        f"No mixture image available for diagnostic config {diagnostic_config_id}."
    )

diagnostic_image_key = str(
    diagnostic_image_df.iloc[0]["source_image"]
)

print("Diagnostic configuration:", diagnostic_config_id)
print("Diagnostic image:", diagnostic_image_key)

display(
    diagnostic_config_df[
        [
            col for col in [
                "selected_config_id",
                "frozen_reference_rank",
                "candidate_source",
                "matrix_family",
                "training_matrix_id",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
                "three_way_lower_threshold",
                "three_way_upper_threshold",
                "pure_test_fn_rate",
                "pure_test_fp_rate",
            ]
            if col in diagnostic_config_df.columns
        ]
    ]
)
display(diagnostic_image_df)


Diagnostic configuration: optuna_object_matrix_0149
Diagnostic image: alm5pea3


,selected_config_id,frozen_reference_rank,candidate_source,matrix_family,training_matrix_id,preprocessing,selected_rule_name,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,pure_test_fn_rate,pure_test_fp_rate
0,optuna_object_matrix_0149,1,04B2_optuna_challenge,object_matrix,object_median,sg_smooth,alternative_chi2_emp_cv,5,0.01,0.5,0.5,0.95,0.0,0.4375


,n,n_target,n_non_target,n_uncertain,uncertain_rate,coverage_rate,target_miss_rate,screening_sensitivity,target_auto_accept_rate,target_uncertain_rate,non_target_false_accept_rate,non_target_auto_reject_rate,non_target_uncertain_rate,decided_tp,decided_fn,decided_fp,decided_tn,decided_accuracy,decided_balanced_accuracy,three_way_score,selected_config_id,source_image,n_true_target_objects
0,38,1,37,9,0.236842,0.763158,1.0,0.0,0.0,0.0,0.0,0.756757,0.243243,0,1,0,28,0.965517,0.5,-19.929232,optuna_object_matrix_0149,alm5pea3,1


In [23]:
essential_figures = {}
essential_figure_paths = {}


def _register_essential_figure(name: str, fig):
    """Display and optionally save one of the few notebook-level figures."""
    if fig is None:
        return

    essential_figures[name] = fig

    if SAVE_ESSENTIAL_FIGURES:
        essential_figure_paths[name] = save_figure_bundle(
            fig,
            FIGURES_DIR / sanitize_filename(name),
            formats=ESSENTIAL_FIGURE_FORMATS,
        )

    if SHOW_ESSENTIAL_FIGURES_INLINE:
        fig.show()


if RUN_ESSENTIAL_VISUALIZATION:
    diagnostic_object_df = mixture_objects_df[
        mixture_objects_df["selected_config_id"]
        .astype(str)
        .eq(diagnostic_config_id)
    ].copy()

    diagnostic_pixel_df = mixture_pixels_df[
        mixture_pixels_df["selected_config_id"]
        .astype(str)
        .eq(diagnostic_config_id)
    ].copy()

    # 1) Object-level binary confusion matrix.
    binary_confusion_df = mixture_object_2way_confusion_df[
        mixture_object_2way_confusion_df["selected_config_id"]
        .astype(str)
        .eq(diagnostic_config_id)
    ].copy()

    if len(binary_confusion_df) > 0:
        fig = plot_binary_confusion_heatmap(
            binary_confusion_df,
            target_class=TARGET_CLASS,
            non_target_label=NON_TARGET_LABEL,
            title=f"Object-level binary confusion — {diagnostic_config_id}",
            show=False,
        )
        _register_essential_figure("01_object_binary_confusion", fig)

    # 2) Object-level 3-way confusion matrix.
    three_way_confusion_df = mixture_object_3way_confusion_df[
        mixture_object_3way_confusion_df["selected_config_id"]
        .astype(str)
        .eq(diagnostic_config_id)
    ].copy()

    if len(three_way_confusion_df) > 0:
        fig = plot_three_way_confusion_heatmap(
            three_way_confusion_df,
            target_class=TARGET_CLASS,
            non_target_label=NON_TARGET_LABEL,
            uncertain_label=UNCERTAIN_LABEL_USED,
            title=f"Object-level 3-way confusion — {diagnostic_config_id}",
            show=False,
        )
        _register_essential_figure("02_object_three_way_confusion", fig)

    # 3) One compact spatial diagnostic for the difficult image.
    diagnostic_radius = diagnostic_config_df.iloc[0].get(
        "position_dilation_radius",
        3,
    )
    diagnostic_radius = (
        3
        if pd.isna(diagnostic_radius)
        else int(diagnostic_radius)
    )

    fig = plot_mixture_diagnostic_panel(
        image_key=diagnostic_image_key,
        image_db=image_db,
        object_db=object_db,
        object_df=diagnostic_object_df,
        pixel_df=diagnostic_pixel_df,
        target_class=TARGET_CLASS,
        dilation_radius=diagnostic_radius,
        crop_to_objects=True,
        padding=5,
        title=(
            f"Mixture diagnostic — {diagnostic_config_id} — "
            f"{diagnostic_image_key}"
        ),
        show=False,
    )
    _register_essential_figure("03_mixture_diagnostic_panel", fig)

    # 4) Sensitivity of the approximate mixture truth for the same config.
    diagnostic_truth_sensitivity_df = (
        mixture_truth_dilation_sensitivity_df[
            mixture_truth_dilation_sensitivity_df["selected_config_id"]
            .astype(str)
            .eq(diagnostic_config_id)
        ].copy()
        if len(mixture_truth_dilation_sensitivity_df) > 0
        else pd.DataFrame()
    )

    if len(diagnostic_truth_sensitivity_df) > 0:
        fig = plot_truth_dilation_sensitivity(
            diagnostic_truth_sensitivity_df,
            radius_col="truth_dilation_radius",
            metric_cols=("fn_rate", "fp_rate", "balanced_accuracy"),
            config_col="selected_config_id",
            title=(
                "Sensitivity to mixture-truth dilation — "
                f"{diagnostic_config_id}"
            ),
            show=False,
        )
        _register_essential_figure("04_truth_dilation_sensitivity", fig)

    print("Essential figures generated:", list(essential_figures))
    if SAVE_ESSENTIAL_FIGURES:
        print("Saved figure files:", essential_figure_paths)
else:
    print("Essential visualisation skipped.")


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Essential figures generated: ['01_object_binary_confusion', '02_object_three_way_confusion', '03_mixture_diagnostic_panel', '04_truth_dilation_sensitivity']


## 11. Save protocol and inspect result files

In [24]:
mixture_application_protocol_df = pd.DataFrame([{
    "db_h5_path": str(DB_H5_PATH),
    "results_04c_dir": str(RESULTS_04C_DIR),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)),

    "target_class": TARGET_CLASS,
    "non_target_label": NON_TARGET_LABEL,
    "uncertain_label": UNCERTAIN_LABEL_USED,
    "reference_classes_json": json.dumps(list(REFERENCE_CLASSES)),

    "frozen_reference_configs_path": str(FROZEN_REFERENCE_CONFIGS_PATH),
    "pure_test_metrics_path": str(PURE_TEST_METRICS_PATH),

    "mixture_final_train_batches_json": json.dumps(MIXTURE_FINAL_TRAIN_BATCHES),
    "mixture_final_train_filters_json": json.dumps(MIXTURE_FINAL_TRAIN_FILTERS, default=str),
    "mixture_filters_json": json.dumps(MIXTURE_FILTERS, default=str),

    "random_state": int(RANDOM_STATE),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "cv_n_splits": int(CV_N_SPLITS) if CV_N_SPLITS is not None else np.nan,
    "cv_group_col": CV_GROUP_COL,

    "apply_fixed_three_way": bool(APPLY_FIXED_THREE_WAY),
    "three_way_threshold_source": "04C_frozen_reference_configs",

    "run_border_diagnostic": bool(RUN_BORDER_DIAGNOSTIC),
    "border_diagnostic_widths_json": json.dumps(BORDER_DIAGNOSTIC_WIDTHS),

    "run_truth_dilation_sensitivity": bool(RUN_TRUTH_DILATION_SENSITIVITY),
    "truth_dilation_radii_json": json.dumps(TRUTH_DILATION_RADII),

    "run_essential_visualization": bool(RUN_ESSENTIAL_VISUALIZATION),
    "show_essential_figures_inline": bool(SHOW_ESSENTIAL_FIGURES_INLINE),
    "save_essential_figures": bool(SAVE_ESSENTIAL_FIGURES),
    "essential_figure_formats_json": json.dumps(list(ESSENTIAL_FIGURE_FORMATS)),
    "diagnostic_config_id": diagnostic_config_id,
    "diagnostic_image_key": diagnostic_image_key,
    "n_essential_figures": int(len(essential_figures)),
    "figures_dir": str(FIGURES_DIR),

    "save_mixture_pixel_tables": bool(SAVE_MIXTURE_PIXEL_TABLES),

    "n_reference_configs": int(len(reference_configs_df)),
    "n_mixture_metrics": int(len(mixture_metrics_df)),
    "n_mixture_objects": int(len(mixture_objects_df)),
    "n_mixture_pixels": int(len(mixture_pixels_df)),
    "n_mixture_refit_errors": int(len(mixture_refit_errors_df)),
    "n_mixture_three_way_metrics": int(len(mixture_three_way_metrics_df)),
    "n_mixture_three_way_by_image": int(len(mixture_three_way_by_image_df)),
    "n_mixture_objects_3way": int(len(mixture_objects_3way_df)),
    "n_mixture_pixels_3way_in_memory": int(len(mixture_pixels_3way_df)),
    "full_pixel_tables_saved": bool(SAVE_MIXTURE_PIXEL_TABLES),
    "n_object_3way_confusion_rows": int(len(mixture_object_3way_confusion_df)),
    "n_pixel_3way_confusion_rows": int(len(mixture_pixel_3way_confusion_df)),
    "n_border_diagnostic_rows": int(len(mixture_border_diagnostic_df)),
    "n_truth_dilation_sensitivity_rows": int(len(mixture_truth_dilation_sensitivity_df)),

    "mixture_metrics_path": str(MIXTURE_METRICS_PATH),
    "mixture_model_summary_path": str(MIXTURE_MODEL_SUMMARY_PATH),
    "mixture_object_predictions_path": str(MIXTURE_OBJECT_PREDICTIONS_PATH),
    "mixture_object_errors_by_image_path": str(MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH),
    "mixture_pixel_errors_by_image_path": str(MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH),
    "mixture_three_way_metrics_path": str(MIXTURE_THREE_WAY_METRICS_PATH),
    "mixture_three_way_by_image_path": str(MIXTURE_THREE_WAY_BY_IMAGE_PATH),
    "mixture_object_predictions_3way_path": str(MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH),
    "mixture_pixel_predictions_3way_path": str(MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH),
    "mixture_object_3way_confusion_path": str(MIXTURE_OBJECT_3WAY_CONFUSION_PATH),
    "mixture_pixel_3way_confusion_path": str(MIXTURE_PIXEL_3WAY_CONFUSION_PATH),
    "mixture_object_simca_diagnostics_path": str(MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH),
    "mixture_pixel_simca_diagnostics_path": str(MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH),
    "mixture_border_diagnostic_path": str(MIXTURE_BORDER_DIAGNOSTIC_PATH),
    "mixture_truth_dilation_sensitivity_path": str(MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH),
    "mixture_application_protocol_path": str(MIXTURE_APPLICATION_PROTOCOL_PATH),
}])

save_parquet(
    mixture_application_protocol_df,
    MIXTURE_APPLICATION_PROTOCOL_PATH,
)

print("Saved mixture application protocol:")
print(MIXTURE_APPLICATION_PROTOCOL_PATH)

display(mixture_application_protocol_df)

Saved mixture application protocol:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_application_protocol.parquet


,db_h5_path,results_04c_dir,results_dir,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,non_target_label,uncertain_label,reference_classes_json,frozen_reference_configs_path,pure_test_metrics_path,mixture_final_train_batches_json,mixture_final_train_filters_json,mixture_filters_json,random_state,replace_balanced_pixels,cv_n_splits,cv_group_col,apply_fixed_three_way,three_way_threshold_source,run_border_diagnostic,border_diagnostic_widths_json,run_truth_dilation_sensitivity,truth_dilation_radii_json,run_essential_visualization,show_essential_figures_inline,save_essential_figures,essential_figure_formats_json,diagnostic_config_id,diagnostic_image_key,n_essential_figures,figures_dir,save_mixture_pixel_tables,n_reference_configs,n_mixture_metrics,n_mixture_objects,n_mixture_pixels,n_mixture_refit_errors,n_mixture_three_way_metrics,n_mixture_three_way_by_image,n_mixture_objects_3way,n_mixture_pixels_3way_in_memory,full_pixel_tables_saved,n_object_3way_confusion_rows,n_pixel_3way_confusion_rows,n_border_diagnostic_rows,n_truth_dilation_sensitivity_rows,mixture_metrics_path,mixture_model_summary_path,mixture_object_predictions_path,mixture_object_errors_by_image_path,mixture_pixel_errors_by_image_path,mixture_three_way_metrics_path,mixture_three_way_by_image_path,mixture_object_predictions_3way_path,mixture_pixel_predictions_3way_path,mixture_object_3way_confusion_path,mixture_pixel_3way_confusion_path,mixture_object_simca_diagnostics_path,mixture_pixel_simca_diagnostics_path,mixture_border_diagnostic_path,mixture_truth_dilation_sensitivity_path,mixture_application_protocol_path
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,almond,uncertain,"[""almond"", ""peanut""]",C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,"[1, 2, 3, 4]","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""mixture""]}",42,False,5,object_id,True,04C_frozen_reference_configs,True,"[1, 2, 3]",True,"[0, 1, 2, 3, 4, 5]",True,True,False,"[""html""]",optuna_object_matrix_0149,alm5pea3,4,C:\Users\alixg\OneDrive - Université Paris-Dau...,False,16,16,11552,1020352,0,16,320,11552,1020352,False,96,96,96,96,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...


In [25]:
print("Result files:")
display(list_result_files(RESULTS_DIR))

Result files:


,file,suffixes,size_mb
0,mixture_pixel_predictions_3way_from_object_dec...,.parquet,59.284425
1,mixture_pixel_simca_q_t2_diagnostics.parquet,.parquet,19.848158
2,figures\04A_object_matrix_0004_object_matrix_o...,.html,7.223015
3,figures\04A_pixel_matrix_0020_pixel_matrix_bal...,.html,7.223014
4,figures\04A_object_matrix_0004_object_matrix_o...,.html,7.222963
...,...,...,...
426,figures\04A_object_matrix_0003_object_matrix_o...,.csv,0.001779
427,figures\04A_pixel_matrix_0018_pixel_matrix_bal...,.csv,0.001774
428,figures\04A_object_matrix_0004_object_matrix_o...,.csv,0.001772
429,figures\04A_pixel_matrix_0021_pixel_matrix_bal...,.csv,0.001769


## 12. Final check

In [26]:
print("05_simca_mixture_application.ipynb completed.")
print()
print("Essential outputs:")
print(" -", MIXTURE_METRICS_PATH)
print(" -", MIXTURE_MODEL_SUMMARY_PATH)
print(" -", MIXTURE_OBJECT_PREDICTIONS_PATH)
print(" -", MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH)
print(" -", MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH)
print(" -", MIXTURE_OBJECT_2WAY_CONFUSION_PATH)
print(" -", MIXTURE_PIXEL_2WAY_CONFUSION_PATH)
print(" -", MIXTURE_THREE_WAY_METRICS_PATH)
print(" -", MIXTURE_THREE_WAY_BY_IMAGE_PATH)
print(" -", MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH)
print(" -", MIXTURE_OBJECT_3WAY_CONFUSION_PATH)
print(" -", MIXTURE_PIXEL_3WAY_CONFUSION_PATH)
print(" -", MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH)
print(" -", MIXTURE_BORDER_DIAGNOSTIC_PATH)
print(" -", MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH)
print(" -", MIXTURE_PARAMETER_TENDENCIES_PATH)
print(" -", MIXTURE_APPLICATION_PROTOCOL_PATH)

if mixture_refit_errors_df is not None and len(mixture_refit_errors_df) > 0:
    print(" -", MIXTURE_REFIT_ERRORS_PATH)

if SAVE_MIXTURE_PIXEL_TABLES:
    print("Large optional pixel outputs:")
    print(" -", MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH)
    print(" -", MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH)
    print(" -", MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH)

if SAVE_ESSENTIAL_FIGURES:
    print("Essential figure directory:")
    print(" -", FIGURES_DIR)

print()
print("The exhaustive figure generation remains delegated to the reporting script.")


05_simca_mixture_application.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_model_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_predictions.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_errors_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_pixel_errors_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_mixture_application_non_noisy_all\mixture_object_2way_confusion.parquet
 - C:\Users\alixg\OneDrive - Université 